# Discrete Math in NLP

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 07.04 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/07_Discrete_Mathematics/05_discrete_math_in_nlp.ipynb)

---

## 🎯 Learning Objective

Use edit distance and sequence alignment to compare strings, and connect partial orders to preference relations.

---

## 📐 Theory

This closing notebook of Module 07 collects four discrete-math ideas that show up constantly
once you look at how NLP systems are actually evaluated and trained: measuring the distance
between two strings, aligning sequences, and ordering things that don't have a single obvious
ranking.

### Edit distance

The **Levenshtein (edit) distance** between two strings is the minimum number of single-character
**insertions**, **deletions**, and **substitutions** needed to turn one string into the other.
It's a genuine **metric**: it's non-negative, symmetric, zero exactly when the strings are
identical, and satisfies the triangle inequality — the same properties a distance in ordinary
geometry has, just defined over strings instead of points in space.

Edit distance has optimal substructure exactly like the tree/grid problems of `07.03`: let
$D_{i,j}$ be the edit distance between the first $i$ characters of string $a$ and the first $j$
characters of string $b$. Either the last characters $a_i, b_j$ match (no edit needed there,
recurse on $D_{i-1,j-1}$), or one of three edits fixes the mismatch:

$$D_{i,j} = \begin{cases} D_{i-1,j-1} & \text{if } a_i = b_j \\ 1 + \min(D_{i-1,j},\ D_{i,j-1},\ D_{i-1,j-1}) & \text{otherwise} \end{cases}$$

($D_{i-1,j}$: delete $a_i$; $D_{i,j-1}$: insert $b_j$; $D_{i-1,j-1}$: substitute $a_i\to b_j$.)
This recurrence, filled in bottom-up, is the **Wagner-Fischer algorithm** — dynamic programming
over a 2D grid, exactly the grid-path structure of `07.03`, giving $O(|a|\cdot|b|)$ time instead
of the exponentially many edit sequences a brute-force search would need to compare.

### Sequence alignment

**Sequence alignment** asks a slightly richer question than a single distance number: *which*
characters of $a$ correspond to *which* characters of $b$? The same DP table that computes edit
distance can be walked backward from $D_{m,n}$ to $D_{0,0}$, and each step of that walk reveals
one aligned pair (a match, a substitution, an insertion, or a deletion) — this is exactly how
tools align DNA sequences, and it's the same idea BLEU-style metrics lean on when scoring how
well a generated sequence lines up with a reference.

### Partial orders and lattices

A relation $\preceq$ on a set is a **partial order** if it's **reflexive** ($x\preceq x$),
**antisymmetric** ($x\preceq y$ and $y\preceq x$ implies $x=y$), and **transitive** ($x\preceq y$
and $y\preceq z$ implies $x\preceq z$) — the same axioms as $\le$ on numbers, minus one: a
partial order does *not* require every pair to be comparable. Two elements $x,y$ are
**incomparable** if neither $x\preceq y$ nor $y\preceq x$ holds — this is the crucial difference
from a **total order** (like $\le$ on real numbers, where every pair *is* comparable). A
**lattice** is a partial order where every pair of elements has both a unique **join** (least
upper bound, generalizing "max") and **meet** (greatest lower bound, generalizing "min") — the
subset relation $\subseteq$ on sets is a classic lattice example, with join = union and meet =
intersection.

### Why "partial," not "total," matters for preferences

Ranking a set of items by a single score gives a total order for free — every pair is
comparable by that score. But when preferences come from *pairwise* human judgments rather than
a shared numeric scale (this response is better than that one, evaluated one pair at a time),
you don't automatically get a total order: you might know $A\succ B$ and $A\succ C$ without ever
having compared $B$ and $C$ directly, leaving $B,C$ genuinely incomparable given the evidence
collected so far. Modeling preferences as a partial order (rather than forcing a total ranking
prematurely) is honest about exactly this gap between "what we've compared" and "what we'd need
to compare to have a full ranking."

---

In [ ]:
# Setup
import difflib
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The edit-distance DP table is a grid where every cell's value depends only on its top,
left, and top-left neighbors — plotting the whole grid, with the optimal alignment path traced
through it, makes both the recurrence and the resulting alignment visible at once.

In [ ]:
# The edit-distance DP grid for "kitten" -> "sitting", with the optimal alignment path traced
# through it by walking backpointers from the bottom-right corner back to (0,0).
def levenshtein_table_with_backpointers(a, b):
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    dp[:, 0], dp[0, :] = np.arange(m + 1), np.arange(n + 1)
    backpointer = {}
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i, j] = dp[i - 1, j - 1]
                backpointer[(i, j)] = (i - 1, j - 1)  # match: no edit, move diagonally
            else:
                candidates = [(dp[i - 1, j] + 1, (i - 1, j)),      # delete a character from a
                               (dp[i, j - 1] + 1, (i, j - 1)),      # insert a character from b
                               (dp[i - 1, j - 1] + 1, (i - 1, j - 1))]  # substitute
                dp[i, j], backpointer[(i, j)] = min(candidates, key=lambda pair: pair[0])
    return dp, backpointer

def trace_alignment_path(backpointer, m, n):
    path, i, j = [(m, n)], m, n
    while (i, j) != (0, 0):
        i, j = backpointer.get((i, j), (max(i - 1, 0), max(j - 1, 0)) if min(i, j) == 0 else (i, j))
        path.append((i, j))
    return path[::-1]

word_a, word_b = "kitten", "sitting"
dp_table, backpointers = levenshtein_table_with_backpointers(word_a, word_b)
alignment_path = trace_alignment_path(backpointers, len(word_a), len(word_b))

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(dp_table, cmap="YlOrRd")
for i in range(dp_table.shape[0]):
    for j in range(dp_table.shape[1]):
        ax.text(j, i, str(dp_table[i, j]), ha="center", va="center", fontsize=9)
ax.plot([p[1] for p in alignment_path], [p[0] for p in alignment_path],
         color="#2A52BE", lw=2.5, marker="o", ms=5)
ax.set_xticks(range(len(word_b) + 1)); ax.set_xticklabels([""] + list(word_b))
ax.set_yticks(range(len(word_a) + 1)); ax.set_yticklabels([""] + list(word_a))
ax.set_title(f'Edit-distance DP grid: "{word_a}" -> "{word_b}"\n(blue path = one optimal alignment)')
plt.tight_layout()
plt.show()

print(f"Edit distance D[{len(word_a)},{len(word_b)}] = {dp_table[-1, -1]}")
print(f"(each cell only ever reads its top/left/top-left neighbors -- the same grid-DP")
print(f"structure as the path-counting problem in 07.03, just with a min instead of a sum)")

## 🐍 Implementation from Scratch

We implement Levenshtein distance two genuinely different ways — bottom-up dynamic programming
and top-down recursion with memoization — and confirm they agree. (Python's `difflib` doesn't
compute true edit distance; its `SequenceMatcher.ratio()` is a related-but-different similarity
score, so we verify against our own second, independent implementation and known textbook
values instead of treating `difflib` as ground truth here.)

In [ ]:
def levenshtein_dp(a, b):
    """Bottom-up dynamic programming, directly implementing the Theory section's recurrence."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i  # deleting all of a[:i] to match an empty prefix of b
    for j in range(n + 1):
        dp[0][j] = j  # inserting all of b[:j] to match an empty prefix of a
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]

def levenshtein_recursive(a, b):
    """Top-down recursion with memoization -- a SEPARATE implementation path (no shared code
    with levenshtein_dp above) to cross-check against, not just the same logic transcribed."""
    from functools import lru_cache

    @lru_cache(maxsize=None)
    def solve(i, j):
        if i == 0:
            return j
        if j == 0:
            return i
        if a[i - 1] == b[j - 1]:
            return solve(i - 1, j - 1)
        return 1 + min(solve(i - 1, j), solve(i, j - 1), solve(i - 1, j - 1))

    result = solve(len(a), len(b))
    solve.cache_clear()
    return result

# Known textbook values, so we're checking against ground truth, not just internal consistency.
known_cases = [("kitten", "sitting", 3), ("flaw", "lawn", 2), ("intention", "execution", 5),
                ("", "abc", 3), ("abc", "abc", 0), ("saturday", "sunday", 3)]
print(f"{'a':<12} {'b':<12} {'dp':>4} {'recursive':>10} {'expected':>9} {'match':>7}")
for a, b, expected in known_cases:
    dp_result = levenshtein_dp(a, b)
    rec_result = levenshtein_recursive(a, b)
    match = dp_result == rec_result == expected
    print(f"{a:<12} {b:<12} {dp_result:>4} {rec_result:>10} {expected:>9} {match!s:>7}")
    assert match

# difflib's ratio() is a DIFFERENT, related quantity (based on matching contiguous blocks, not
# minimum edit operations) -- worth showing explicitly so the distinction is never confused.
a, b = "intention", "execution"
edit_dist = levenshtein_dp(a, b)
difflib_ratio = difflib.SequenceMatcher(None, a, b).ratio()
print(f"\n'{a}' vs '{b}': Levenshtein distance = {edit_dist}, difflib ratio() = {difflib_ratio:.3f}")
print("These measure genuinely different things -- ratio() is NOT 1 minus normalized edit distance.")

## 🤖 Why This Matters for AI

Two direct connections. First, evaluating generated text against a reference is fundamentally a
string/sequence comparison problem: **word error rate** (speech recognition) is literally edit
distance divided by reference length, and **BLEU score** (machine translation) generalizes the
same "how much overlap" idea to n-grams — computing precision over matching n-grams of several
lengths and combining them. Second, **RLHF (Reinforcement Learning from Human Feedback)** trains
on *pairwise* preference judgments ("response A is better than response B") rather than absolute
scores, which is exactly a partial order over model outputs, not a total order — the reward
model's job is to find a scoring function *consistent* with the observed pairwise comparisons,
even though many pairs were never directly compared.

In [ ]:
# --- Part 1: edit distance and n-gram precision as text-generation evaluation metrics ---
def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def modified_precision(candidate, reference, n):
    """BLEU's core building block: what fraction of the candidate's n-grams appear in the
    reference, capped so repeating a common n-gram can't inflate the score for free."""
    from collections import Counter
    cand_ngrams, ref_ngrams = Counter(ngrams(candidate, n)), Counter(ngrams(reference, n))
    if not cand_ngrams:
        return 0.0
    overlap = sum(min(count, ref_ngrams[gram]) for gram, count in cand_ngrams.items())
    return overlap / sum(cand_ngrams.values())

def bleu_score(candidate, reference, max_n=4):
    """A simplified BLEU: geometric mean of n-gram precisions (n=1..max_n) times a brevity
    penalty that discourages generating suspiciously short candidates to game precision."""
    precisions = [modified_precision(candidate, reference, n) for n in range(1, max_n + 1)]
    geometric_mean = 0.0 if min(precisions) == 0 else np.exp(np.mean(np.log(precisions)))
    brevity_penalty = min(1.0, np.exp(1 - len(reference) / len(candidate))) if candidate else 0.0
    return geometric_mean * brevity_penalty, precisions

reference_sentence = "the cat sat on the mat".split()
candidate_sentences = {
    "identical": "the cat sat on the mat".split(),
    "one word off": "the cat sat on a mat".split(),
    "two words off": "a cat sat on a mat".split(),
    "unrelated": "quantum entanglement defies classical intuition physics".split(),
}

print(f"{'candidate':<15} {'word edit distance':>19} {'BLEU':>8}   1-4gram precisions")
for label, candidate in candidate_sentences.items():
    edit_dist = levenshtein_dp(reference_sentence, candidate)  # word-level, not character-level
    score, precisions = bleu_score(candidate, reference_sentence)
    print(f"{label:<15} {edit_dist:>19} {score:>8.4f}   {[round(p, 2) for p in precisions]}")
print("\nEdit distance degrades gradually (0, 1, 2, 6 operations) as the candidate drifts from")
print("the reference. BLEU is much less forgiving on short sentences: it HITS ZERO as soon as")
print("even one n-gram order has zero overlap (the geometric mean of any list containing a 0 is")
print("0) -- 'two words off' already loses every 4-gram match, so BLEU floors at exactly 0.0")
print("despite still sharing most individual words with the reference. This is a well-known")
print("brittleness of unsmoothed BLEU on short text, and exactly why smoothed variants exist.")

# --- Part 2: pairwise preferences (RLHF) form a partial order, not automatically a total one ---
responses = ["A", "B", "C", "D", "E"]
pairwise_judgments = [("A", "B"), ("B", "C"), ("A", "D"), ("D", "E")]  # (winner, loser); NOT all pairs judged

def transitive_closure(pairs, items):
    """Everything x is known to beat, directly or by chaining through other judgments."""
    reachable = {x: set() for x in items}
    for winner, loser in pairs:
        reachable[winner].add(loser)
    changed = True
    while changed:
        changed = False
        for x in items:
            expanded = set().union(*(reachable[y] for y in reachable[x])) if reachable[x] else set()
            if not expanded.issubset(reachable[x]):
                reachable[x] |= expanded
                changed = True
    return reachable

reachable = transitive_closure(pairwise_judgments, responses)
print(f"\nFrom {len(pairwise_judgments)} pairwise judgments over {len(responses)} responses:")
for x in responses:
    print(f"  {x} is known to beat: {sorted(reachable[x]) if reachable[x] else '(nothing directly comparable)'}")

incomparable_pairs = [(x, y) for i, x in enumerate(responses) for y in responses[i + 1:]
                       if y not in reachable[x] and x not in reachable[y]]
print(f"\nIncomparable pairs (no direct or chained evidence either way): {incomparable_pairs}")
print(f"This is a genuine PARTIAL order, not a total one -- {len(incomparable_pairs)} of the "
      f"{len(responses) * (len(responses) - 1) // 2} possible pairs have no established preference.")

has_cycle = any(y in reachable[x] and x in reachable[y]
                 for i, x in enumerate(responses) for y in responses[i + 1:])
print(f"Cycle detected (would mean contradictory/inconsistent human judgments): {has_cycle}")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, fill in the edit-distance DP table for `"cat"` vs `"cot"`. Confirm the answer is 1
   (a single substitution), and verify with `levenshtein_dp`.

<details>
<summary>💡 Solution</summary>

The filled table (rows indexed by prefixes of `"cat"`, columns by prefixes of `"cot"`) is:
```
      ""  c  o  t
  ""   0  1  2  3
  c    1  0  1  2
  a    2  1  1  2
  t    3  2  2  1
```
$D_{3,3}=1$: `"cat"` and `"cot"` differ in exactly one position (`a`→`o`), a single substitution,
matching `levenshtein_dp("cat","cot")==1`.

</details>

### 💭 UNDERSTAND
2. Edit distance is a genuine metric, so it must satisfy the triangle inequality
   $D(a,c) \le D(a,b) + D(b,c)$ for any three strings. Without computing anything, explain in
   one or two sentences *why* this must hold from the definition of edit distance as a minimum
   number of operations (hint: think about what happens if you compose an optimal edit sequence
   from $a$ to $b$ with an optimal one from $b$ to $c$).

<details>
<summary>💡 Solution</summary>

Take the optimal (shortest) sequence of edits transforming $a$ into $b$ (length $D(a,b)$), and
immediately follow it with the optimal sequence transforming $b$ into $c$ (length $D(b,c)$).
Concatenating these two sequences is *some* valid way to transform $a$ into $c$, using
$D(a,b)+D(b,c)$ operations total. Since $D(a,c)$ is defined as the *minimum* over ALL possible
ways to transform $a$ into $c$, and this concatenated sequence is one such way (not necessarily
the best one), $D(a,c)$ can be no larger than this particular sequence's length — i.e.
$D(a,c)\le D(a,b)+D(b,c)$. The inequality isn't an extra assumption; it falls directly out of
"minimum over all sequences" being at most the length of any *specific* sequence you can name.

</details>

### ✏️ DERIVE
3. Derive the recurrence's correctness for the substitution case: show that if $a_i \ne b_j$,
   the minimum-edit transformation of $a_{1:i}$ into $b_{1:j}$ is achieved by one of exactly the
   three options the recurrence considers (delete $a_i$, insert $b_j$, or substitute $a_i \to
   b_j$) — i.e. explain why no fourth kind of first-move could ever do better.

<details>
<summary>💡 Solution outline</summary>

Consider any sequence of edits transforming $a_{1:i}$ into $b_{1:j}$, and look at what happens
to the very last character of the current working string, $a_i$, and the very last character
we need to produce, $b_j$ (with $a_i \ne b_j$). Exactly one of three things can be true about
the *final* edit operation touching position $i$ of $a$ or position $j$ of $b$: (1) $a_i$ is
deleted and never contributes to the output — this reduces the problem to transforming
$a_{1:i-1}$ into $b_{1:j}$, cost $1+D_{i-1,j}$; (2) $b_j$ is produced by an insertion not coming
from $a_i$ — this reduces to transforming $a_{1:i}$ into $b_{1:j-1}$, cost $1+D_{i,j-1}$; (3)
$a_i$ is directly turned into $b_j$ via substitution (since they're unequal, this costs exactly
1) — reducing to transforming $a_{1:i-1}$ into $b_{1:j-1}$, cost $1+D_{i-1,j-1}$. Every possible
edit sequence must fall into one of these three cases when you look at how position $i$ of $a$
is finally accounted for — there is no fourth option, because $a_i$ is either deleted, matched
to a specific character of $b$ via substitution, or left alone while something is inserted
before/after it (which is exactly case 2, viewed from $b_j$'s side). Taking the minimum over the
three cases is therefore not just *an* upper bound but the *exact* minimum, which is precisely
$D_{i,j} = 1+\min(D_{i-1,j}, D_{i,j-1}, D_{i-1,j-1})$.

</details>

### 🐍 IMPLEMENT
4. Modify `levenshtein_dp` to also return the actual sequence of edit operations (not just the
   count), by storing which of the three cases (match, delete, insert, substitute) was chosen
   at each cell, then walking backpointers from $(m,n)$ to $(0,0)$ — the same backpointer idea
   used in the Visual Intuition section. Test it on `"kitten"` vs `"sitting"`.

<details>
<summary>✅ How to know you're right</summary>

The number of non-match operations in your returned sequence should equal exactly
`levenshtein_dp("kitten", "sitting")` = 3 (confirmed by the notebook's own `known_cases` table).
Applying your returned operations one by one to `"kitten"` (substituting/inserting/deleting as
each operation specifies) should, after all operations, produce exactly `"sitting"` — this
"replay the operations and check you land on the target string" test is a stronger check than
just counting operations, since it catches bugs where the *count* is right but the specific
operations or their order don't actually work.

</details>

### 🤖 APPLY
5. Word error rate (WER), mentioned in "Why This Matters for AI," is word-level edit distance
   divided by reference length. Using `levenshtein_dp` on lists of words (not characters, i.e.
   `"the cat sat".split()` rather than the raw string), compute the WER for a reference
   transcript `"the quick brown fox jumps"` against two candidate ASR outputs: (a) `"the quick
   brown fox jumped"` (one word wrong) and (b) `"the quick brown fox jumps over"` (one word
   inserted). Confirm both candidates get a nonzero WER despite differing in exactly one word
   each, and explain why WER treats a wrong word and an extra word as costing the same amount.

<details>
<summary>✅ What you should observe</summary>

Both candidates should give word-level edit distance 1 (one substitution for (a), one insertion
for (b)), and since the reference has 5 words, both give $\text{WER}=1/5=0.20$ — identical WER
despite one error being a substitution and the other an insertion. This matches the Levenshtein
recurrence directly: it treats every one of its three operation types (insert, delete,
substitute) as costing exactly 1, with no built-in notion that a wrong word is "worse" or
"better" than an extra or missing one — WER inherits this uniform-cost assumption from edit
distance wholesale, which is a real (if often reasonable) modeling choice, not a law of nature.

</details>

### 🚀 CHALLENGE
6. Add a sixth response `F` to the preference-learning demo in the AI section, with a single
   new judgment `("F", "C")`. Recompute the transitive closure and the incomparable-pairs list.
   Then add one more judgment, `("E", "F")`, creating a full chain from `A` down to `F`. How many
   incomparable pairs remain? In your own words, explain why an RLHF reward model trained on a
   sparse set of pairwise comparisons (as real human-feedback datasets always are, since
   comparing every possible pair of outputs is infeasible) has to *generalize* — fitting a
   single scalar score to every response such that it's consistent with the observed
   comparisons, even for pairs that were never directly judged.

<details>
<summary>🔍 What a strong answer covers</summary>

With the original 4 judgments over 5 responses, 4 of the $\binom{5}{2}=10$ possible pairs are
incomparable: `(B,D)`, `(B,E)`, `(C,D)`, `(C,E)`. Adding `F` with only `("F","C")` and
recomputing over all $\binom{6}{2}=15$ pairs actually *increases* the incomparable count to 8
(the new pairs involving `F` — `(A,F)`, `(B,F)`, `(D,F)`, `(E,F)` — are all incomparable too,
since `F`'s only known relationship is to `C`). Adding `("E","F")` then closes many of those
gaps at once: `D` beats `E` beats `F` (and `F` beats `C`) chains together, so `D` and `E` now
both transitively beat `F` and `C`, dropping the incomparable count to 3 — only `(B,D)`, `(B,E)`,
`(B,F)` remain, all because `B`'s only known relationships are beating `C` and losing to `A`,
with no evideence connecting `B` to the `D`-`E`-`F` chain at all. A strong answer explains
generalization concretely: a reward model can't just memorize "A beats B, B beats C, ..." as a
lookup table and correctly answer queries about incomparable pairs like `(B,D)`, since no amount
of transitive-closure reasoning over the *observed* judgments resolves them — the model has to
learn some notion of "what makes a response good" from features of the responses themselves
(not just their identities) so that it can produce a sensible, consistent scalar score even for
pairs it never saw compared, which is exactly the generalization gap a lookup table (or a plain
transitive closure) cannot close but a trained scoring function can attempt to.

---

## 📚 Further Reading
- Papineni et al., ["BLEU: a Method for Automatic Evaluation of Machine Translation"](https://aclanthology.org/P02-1040/)
- Navarro, ["A Guided Tour to Approximate String Matching"](https://dl.acm.org/doi/10.1145/375360.375365)
- Ouyang et al., ["Training language models to follow instructions with human feedback"](https://arxiv.org/abs/2203.02155) (InstructGPT / RLHF)

---

*Next notebook: [Inner Product Spaces](../08_Advanced_Linear_Algebra/01_inner_product_spaces.ipynb)*